# Spam Email Classifier - Exploratory Data Analysis & Model Prototyping
**Author**: Senior Machine Learning Engineer  
**Project**: End-to-End Spam Classification System

This notebook guides you step-by-step through:
1. Loading & Inspecting the Spam Dataset
2. Data Cleaning & Duplicate Handling
3. NLP Text Preprocessing (Tokenization, Stopword Removal, Stemming)
4. Feature Extraction via TF-IDF Vectorization
5. Model Training & Benchmarking (MultinomialNB, Logistic Regression, Random Forest, SVM)
6. Metric Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. Model Selection & Exporting Artifacts

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

## 1. Load Processed Clean Dataset

In [ ]:
dataset_path = '../data/processed/cleaned_dataset.csv'
df = pd.read_csv(dataset_path)
print("Shape:", df.shape)
df.head()

## 2. Text Preprocessing Function

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_email(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = word_tokenize(text)
    cleaned = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(cleaned)

df['clean_text'] = df['text'].apply(clean_email)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df[['text', 'clean_text', 'label', 'target']].head()

## 3. Train / Test Split & TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(f"X_train shape: {X_train_vec.shape}")
print(f"X_test shape:  {X_test_vec.shape}")

## 4. Train & Benchmark 4 Classifiers

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.2),
    'Logistic Regression': LogisticRegression(C=2.0, max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=25, random_state=42, n_jobs=-1, class_weight='balanced'),
    'Support Vector Machine': CalibratedClassifierCV(
        estimator=SVC(kernel='linear', C=1.0, random_state=42, class_weight='balanced'), cv=3
    )
}

results = []
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

results_df = pd.DataFrame(results)
results_df